# 第 14 章 · 单刚体凸 MPC 与 WBC：四足怎么「算」出每只脚该推多大力

配套讲义：仓库里 `robot-motion-rl/notes/lessons/14-convex-mpc.md`（附录最后一节"代码对照"）

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的 Python 写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。只用 NumPy，不需要 GPU。练习题不在这里算，先在讲义里自己做。

In [ ]:
import numpy as np

np.set_printoptions(precision=4, suppress=True)  # 打印保留 4 位小数，不用科学计数法
print("numpy", np.__version__)

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。

（质量、μ、雅可比都是编的）

| 讲义 | 公式 | 写法 | 跑出来 |
|---|---|---|---|
| 第二步 竖直力 | mg / n | `stance_normal_force(9, 2)` | 44.1 N |
| 第二步 摩擦锥 | μ·f_z | `max_horizontal_force(44.1, 0.6)` | 26.46 N；μg = 5.88 |
| 第三步 滚动优化 | 规划 10 步、执行第 1 步 | `receding_horizon(0, 1, 9, 10, 0.03, 52.92, 10)` | 0、0.176、0.353 … → 1.0 |
| 第四步 WBC | Jᵀf | `leg_torque(J, [0, 44.1])` | (13.23, 8.82) |

数字是编的，质量 9 kg 接近 MIT Mini Cheetah。

## 1. 先把要用的东西定义好

这几格只是定义，运行后没有输出也正常。

In [ ]:
G = 9.8

**函数 `stance_normal_force`**：匀速站立时每只着地脚分担的竖直力：mg / n。

In [ ]:
def stance_normal_force(mass, n_feet):
    """匀速站立时每只着地脚分担的竖直力：mg / n。"""
    return mass * G / n_feet

**函数 `max_horizontal_force`**：摩擦锥（线性化成金字塔）：|f_x| ≤ μ·f_z。

In [ ]:
def max_horizontal_force(fz, mu):
    """摩擦锥（线性化成金字塔）：|f_x| ≤ μ·f_z。"""
    return mu * fz

**函数 `max_accel`**：所有脚都用满摩擦时的最大水平加速度：μ·g（与质量无关）。

In [ ]:
def max_accel(mu):
    """所有脚都用满摩擦时的最大水平加速度：μ·g（与质量无关）。"""
    return mu * G

**函数 `mpc_plan`**：一维速度 MPC：选 horizon 步水平力，min Σ(v_k − v_ref)² + smooth·Σf²，再把力截到 ±f_max。

In [ ]:
def mpc_plan(v0, v_ref, mass, horizon, dt, f_max, smooth=1e-6):
    """一维速度 MPC：选 horizon 步水平力，min Σ(v_k − v_ref)² + smooth·Σf²，再把力截到 ±f_max。
    动力学 v_{k+1} = v_k + f_k·dt/m 是线性的，用最小二乘求解。"""
    lower = np.tril(np.ones((horizon, horizon))) * dt / mass
    a = np.vstack([lower, np.sqrt(smooth) * np.eye(horizon)])
    y = np.concatenate([np.full(horizon, v_ref - v0), np.zeros(horizon)])
    f = np.linalg.lstsq(a, y, rcond=None)[0]
    return np.clip(f, -f_max, f_max)

**函数 `receding_horizon`**：滚动优化：每次规划 horizon 步、只执行第一步，返回速度轨迹。

In [ ]:
def receding_horizon(v0, v_ref, mass, horizon, dt, f_max, steps):
    """滚动优化：每次规划 horizon 步、只执行第一步，返回速度轨迹。"""
    v, vs = v0, [v0]
    for _ in range(steps):
        f = mpc_plan(v, v_ref, mass, horizon, dt, f_max)[0]
        v += f * dt / mass
        vs.append(v)
    return np.array(vs)

**函数 `leg_torque`**：WBC 的核心一步：着地脚要出力 f，关节力矩 τ = Jᵀ·f。

In [ ]:
def leg_torque(jacobian, force):
    """WBC 的核心一步：着地脚要出力 f，关节力矩 τ = Jᵀ·f。"""
    return np.asarray(jacobian).T @ np.asarray(force)

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("\n第二步 地面反力与摩擦锥")
fz = stance_normal_force(9.0, 2)
print(f"9 kg、对角两脚着地：每脚 f_z = {fz:.1f} N；μ=0.6 时 |f_x| ≤ {max_horizontal_force(fz, 0.6):.2f} N")
print(f"最大水平加速度 μg = {max_accel(0.6):.2f} m/s²")

In [ ]:
print("\n第三步 滚动优化")
vs = receding_horizon(0.0, 1.0, 9.0, 10, 0.03, 2 * max_horizontal_force(fz, 0.6), 10)
print("速度（每 0.03 s）：", np.round(vs, 3))

In [ ]:
print("\n第四步 WBC：τ = Jᵀf")
j = [[0.0, -0.4], [0.3, 0.2]]
print("J =", j, "，f = [0, 44.1] → τ =", np.round(leg_torque(j, [0.0, 44.1]), 3))

## 3. 代码里才看得到的

- 真正的凸 MPC 把摩擦锥写进 QP 的约束里一起解；代码为了只用 NumPy，先解无约束最小二乘再截断，结果在这个一维例子里差不多，但多维时可能不是最优。
- 两脚着地，所以每步的总水平力上限是 2 × 26.46 = 52.92 N，这才对应 μg 的加速度。

## 4. 自己改一改

- 把目标速度改成 2 m/s，看要多少步才能到、哪几步顶在摩擦锥上。
- 把 μ 改成 0.2（冰面），看多少步才能到 1 m/s。

在下面的空格子里写，或者直接改上面的格子。

In [ ]:
# 在这里写你的代码

## 口诀

> 机身当砖头，只算每只脚推多大，问题变凸；
> 只推不拉、不能太斜，摩擦锥把加速度锁在 μg；
> 看一段、走一步、再重看，三十赫兹滚着解；
> 力乘雅可比转置变力矩，WBC 五百赫兹顾全身。